In [1]:
import pandas as pd
from tqdm import tqdm
import json
import requests 
from bs4 import BeautifulSoup
import ollama

Big Picture Architecture (what I am building)

I am building a data generation pipeline:
1. Load my dataset
2. Analyze structure (patterns + distributions)
3. Build rule dictionaries (the interpretive logic)
4. Sample structured prompts
5. Use LLM to generate batches
6. Validate outputs (schema + logic)
7. Append + repeat until 500-1000 rows

CSV->Pattern Extraction->Rules->LLM Generator->Validator->Final Dataset

In [2]:
nfl_df = pd.read_csv("nfl_media.csv")
nfl_df.head()

,ID,Player Name,Player Status,Year of Event,Statement Made,Event Description,Platform,Event Type,Media Tone,Post Status,Media Coverage,Sources,URL,Unnamed: 13,Unnamed: 14
0,1.0,Jason Kelce,Retired,2025.0,Man I love the 4th...we all share in common th...,Instagram post celebrating July 4th sparked ba...,Instagram,Controversy,Negative,Active,High,"Fox News, US Weekly, Yahoo Sports",https://www.yahoo.com/entertainment/articles/j...,NaN,NaN
1,2.0,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NaN,NaN
2,3.0,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,NaN,NaN
3,4.0,Travis Kelce,Playing,2024.0,happy easter...#shoutout to Jesus for takin on...,Wishing everyone a Happy Easter while making j...,Twitter/X,Controversy,Mixed,Active,High,"Yahoo Sports, E! News, People",https://www.yahoo.com/entertainment/articles/t...,NaN,NaN
4,5.0,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,NaN,NaN


In [3]:
def get_website_text(url):
    response = requests.get(url)
    print(response.status_code)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text)
        website_text = soup.get_text()
        if (website_text != None):
            return website_text
    else:
        print("URL did not work")

In [4]:
test_nfl_df = nfl_df[0:75].copy()
test_nfl_df['website_text'] = test_nfl_df['URL'].apply(get_website_text)

429
URL did not work
200
200
429
URL did not work
200
200
406
URL did not work
406
URL did not work
401
URL did not work
200
200
200
200
403
URL did not work
202
URL did not work
403
URL did not work
429
URL did not work
200
200
200
429
URL did not work
200
429
URL did not work
200
200
429
URL did not work
200
200
200
200
200
200
200
200
200
403
URL did not work
403
URL did not work
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
406
URL did not work
200
200
200
200
406
URL did not work
200
403
URL did not work
200
403
URL did not work
200
200
200
200
200


In [5]:
test_nfl_df['website_text'].notna().value_counts()

website_text
True     57
False    18
Name: count, dtype: int64

There are 55 out of the 75 events that have code 200 (i.e. worked)

In [6]:
test_nfl_df

,ID,Player Name,Player Status,Year of Event,Statement Made,Event Description,Platform,Event Type,Media Tone,Post Status,Media Coverage,Sources,URL,Unnamed: 13,Unnamed: 14,website_text
0,1.0,Jason Kelce,Retired,2025.0,Man I love the 4th...we all share in common th...,Instagram post celebrating July 4th sparked ba...,Instagram,Controversy,Negative,Active,High,"Fox News, US Weekly, Yahoo Sports",https://www.yahoo.com/entertainment/articles/j...,NaN,NaN,NaN
1,2.0,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NaN,NaN,NFL legend Warren Sapp calls Texas ‘fake footb...
2,3.0,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,NaN,NaN,\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\...
3,4.0,Travis Kelce,Playing,2024.0,happy easter...#shoutout to Jesus for takin on...,Wishing everyone a Happy Easter while making j...,Twitter/X,Controversy,Mixed,Active,High,"Yahoo Sports, E! News, People",https://www.yahoo.com/entertainment/articles/t...,NaN,NaN,NaN
4,5.0,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,NaN,NaN,Deion Sanders Eats Crow After His Shedeur Sand...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,71.0,Dak Prescott,Playing,2020.0,"I think it's huge to talk, I think it's huge t...",Opens about mental health struggles during the...,In Depth With Granham Bensinger in-depth inter...,Support,Positive,Available,Medium,The New York Times,https://www.nytimes.com/athletic/2058072/2020/...,NaN,NaN,‘It saves lives’: Dak Prescott opens up on men...
71,72.0,George Kittle,Playing,2024.0,They had us in the first half not gonna lie,"When advancing to the Super Bowl, Kittle broug...",Postgame Interview,Humor,Positive,Available,Low,Larry Brown Sports,https://ninerswire.usatoday.com/story/sports/n...,NaN,NaN,George Kittle used meme to speak 49ers NFC cha...
72,73.0,Aaron Rodgers,Playing,2021.0,"To anyboady who felt misled by those comments,...",Looked back on comments made previously about ...,The Pat McAfee Show,Controversy,Negative,Available,Medium,"ESPN, the New York Times",https://www.nfl.com/news/aaron-rodgers-full-re...,NaN,NaN,Aaron Rodgers takes 'full responsibility' for ...
73,74.0,CeeDee Lamb,Playing,2025.0,"Just pay the man, what you owe em. No need for...",Publicly backed up teammate Micah Parsons with...,Twitter/X,Support,Positive,Unavailable,Medium,Yahoo Sports,https://sports.yahoo.com/article/ceedee-lamb-t...,NaN,NaN,"CeeDee Lamb Tells Jerry Jones, Cowboys to ‘Pay..."


In [7]:
test_nfl_df_true = test_nfl_df.dropna(subset='website_text')

In [8]:
test_nfl_df_true.head()

,ID,Player Name,Player Status,Year of Event,Statement Made,Event Description,Platform,Event Type,Media Tone,Post Status,Media Coverage,Sources,URL,Unnamed: 13,Unnamed: 14,website_text
1,2.0,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NaN,NaN,NFL legend Warren Sapp calls Texas ‘fake footb...
2,3.0,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,NaN,NaN,\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\...
4,5.0,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,NaN,NaN,Deion Sanders Eats Crow After His Shedeur Sand...
5,6.0,Rashard Mendenhall,Retired,2023.0,I'm sick of average white guys commenting on f...,Made a racist tweet about white NFL playerss,Twitter/X,Controversy,Negative,Deleted,Medium,"ESPN, CBS Sports",https://www.outkick.com/sports/sports-media-si...,NaN,NaN,\n\n\n\nSports Media Silent On Rashard Mendenh...
9,10.0,Princely Umanmielen,Playing,2025.0,#justiceforyounghoekoo,"""Funny"" response after Facon's loss after bein...",Twitter/X,Humor,Positive,Active,Low,PanthersWire,https://sports.yahoo.com/article/panthers-olb-...,NaN,NaN,Panthers OLB Princely Umanmielen explains his ...


In [9]:
test_nfl_df_true = test_nfl_df_true.drop(columns=['Unnamed: 13', 'Unnamed: 14'])

In [10]:
test_nfl_df_true.website_text.str.len()

1     11609
2     20769
4     10569
5      4788
9     10159
10    10346
11    13386
12       53
17    10177
18    10824
19    10983
21    11389
23    10163
24    10301
26    11862
27    10073
28     3128
29    11240
30     8221
31    10830
32    10280
33     9660
34    10905
37    11254
38    10501
39    10643
40    13149
41     6316
42    14328
43    28148
44       24
45     7503
46    17059
47    11005
48    11669
49    10600
50    11840
51    19548
52    10562
53    13262
54    12631
55    20703
56    10129
57    10524
58    14545
59    11310
61    14176
62     6685
63    10194
64     9941
66    16998
68     7224
70    15046
71     3980
72    11884
73    12849
74    10183
Name: website_text, dtype: int64

Use Ollama (3.1) - so can use LLM programmatically

In [11]:
import ollama

response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Say hello in one sentence.'}]
)
print(response['message']['content'])

Hello!


In [17]:
event_df = test_nfl_df.copy()

In [18]:
all_events = event_df['Event Type'].unique().tolist()
all_events

['Controversy', 'Apology', 'Support', 'Humor', 'Reflection']

In [19]:
event_df[event_df['Event Type'].isna()]

,ID,Player Name,Player Status,Year of Event,Statement Made,Event Description,Platform,Event Type,Media Tone,Post Status,Media Coverage,Sources,URL,Unnamed: 13,Unnamed: 14,website_text


Event types

In [ ]:
import ollama

# Build the event list from our actual dataset
event_list = '\n'.join(f'- {g}' for g in sorted(all_events))

def classify_cold(row):
    """Zero-shot genre classification — text only, no prior signals."""
    url = str(row.get('URL', ''))
    website_text = str(row.get('website_text', ''))
    if pd.isna(website_text) or len(website_text) < 200:
        return None
    passage = website_text[:2000]

    prompt = f"""You are classifying NFL-related events into event types.

Source URL: {url}

Website Text: {passage}

Classify this event into EXACTLY ONE of the following event types:
{event_list}

Respond with ONLY the event name from the list above. No explanation, no punctuation."""

    try:
        response = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"Error for {row['title']}: {e}")
        return None

In [96]:
# Run LLM classification
test_nfl_df_true['llm_pred_event'] = test_nfl_df_true.apply(classify_cold, axis=1)

In [97]:
y_true = test_nfl_df_true['Event Type']
y_pred = test_nfl_df_true['llm_pred_event']

In [98]:
test_nfl_df_true[['Event Type', 'llm_pred_event']].head(10)

,Event Type,llm_pred_event
1,controversy,Medium
2,apology,Medium
4,controversy,Medium
5,controversy,Medium
9,humor,Low
10,humor,Low
11,support,Medium
12,controversy,NaN
17,controversy,Medium
18,controversy,Medium


Some NaN values for llm_pred_event

In [99]:
test_nfl_df_true['llm_pred_event'] = (
    test_nfl_df_true['llm_pred_event']
    .str.strip()
    .str.lower()
)

test_nfl_df_true['Event Type'] = (
    test_nfl_df_true['Event Type']
    .str.strip()
    .str.lower()
)

In [100]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Drop missing predictions
df_eval = test_nfl_df_true.dropna(subset=['llm_pred_event'])

accuracy = accuracy_score(df_eval['Event Type'], df_eval['llm_pred_event'])
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(df_eval['Event Type'], df_eval['llm_pred_event']))

print("\nConfusion Matrix:")
print(confusion_matrix(df_eval['Event Type'], df_eval['llm_pred_event']))

Accuracy: 0.0

Classification Report:
              precision    recall  f1-score   support

     apology       0.00      0.00      0.00       2.0
 controversy       0.00      0.00      0.00      26.0
       humor       0.00      0.00      0.00      10.0
         low       0.00      0.00      0.00       0.0
      medium       0.00      0.00      0.00       0.0
  reflection       0.00      0.00      0.00       4.0
     support       0.00      0.00      0.00      13.0

    accuracy                           0.00      55.0
   macro avg       0.00      0.00      0.00      55.0
weighted avg       0.00      0.00      0.00      55.0


Confusion Matrix:
[[ 0  0  0  0  2  0  0]
 [ 0  0  0  0 26  0  0]
 [ 0  0  0  2  8  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  4  0  0]
 [ 0  0  0  0 13  0  0]]


c:\Users\mayab\.virtualenvs\is310-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mayab\.virtualenvs\is310-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mayab\.virtualenvs\is310-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0

In [101]:
errors = df_eval[df_eval['Event Type'] != df_eval['llm_pred_event']]

errors[['Event Type', 'llm_pred_event', 'URL']].head(10)

,Event Type,llm_pred_event,URL
1,controversy,medium,https://sports.yahoo.com/articles/nfl-legend-w...
2,apology,medium,https://www.cnn.com/2025/12/18/sport/football-...
4,controversy,medium,https://sports.yahoo.com/article/deion-sanders...
5,controversy,medium,https://www.outkick.com/sports/sports-media-si...
9,humor,low,https://sports.yahoo.com/article/panthers-olb-...
10,humor,low,https://sports.yahoo.com/article/ben-dinucci-n...
11,support,medium,https://www.nfl.com/news/nfl-trailblazer-carl-...
17,controversy,medium,https://sports.yahoo.com/articles/kyle-pitts-c...
18,controversy,medium,https://sports.yahoo.com/articles/shipped-off-...
19,controversy,medium,https://www.usmagazine.com/entertainment/news/...


In [ ]:
df_eval['match'] = df_eval['Event Type'] == df_eval['llm_pred_event']
agreement_rate = df_eval['match'].mean()

print("Agreement rate:", agreement_rate)

Agreement rate: 0.6181818181818182


In [54]:
errors['Event Type'].value_counts()

Event Type
support        10
humor           7
apology         2
reflection      1
controversy     1
Name: count, dtype: int64

In [55]:
nfl_df.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL', 'Unnamed: 13',
       'Unnamed: 14'],
      dtype='str')

Media Tones

In [58]:
media_df_tone = test_nfl_df_true.copy()

In [ ]:
all_media_tones = media_df_tone['Media Tone'].unique().tolist()

In [ ]:
# Build the media tone list from our actual dataset
media_tones_list = '\n'.join(f'- {g}' for g in sorted(all_media_tones))

def classify_cold(row):
    """Zero-shot genre classification — text only, no prior signals."""
    url = str(row.get('URL', ''))
    website_text = str(row.get('website_text', ''))
    if pd.isna(website_text) or len(website_text) < 200:
        return None
    passage = website_text[:2000]

    prompt = f"""You are classifying NFL-related events into media tones.

Source URL: {url}

Website Text: {passage}

Classify this event into EXACTLY ONE of the following media tones:
{media_tones_list}

Respond with ONLY the media tone from the list above. No explanation, no punctuation."""

    try:
        response = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': prompt}]
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"Error for {row['title']}: {e}")
        return None

In [70]:
# Run LLM classification
test_nfl_df_true['llm_pred_tones'] = test_nfl_df_true.apply(classify_cold, axis=1)

In [71]:
y_true = test_nfl_df_true['Media Tone']
y_pred = test_nfl_df_true['llm_pred_tones']

In [72]:
test_nfl_df_true[['Media Tone', 'llm_pred_tones']].head(10)

,Media Tone,llm_pred_tones
1,Negative,Negative
2,Negative,Negative
4,Negative,Positive
5,Negative,Negative
9,Positive,Positive
10,Positive,Negative
11,Positive,Positive
12,Negative,NaN
17,Negative,Negative
18,Negative,Negative


In [73]:
test_nfl_df_true['llm_pred_tones'] = (
    test_nfl_df_true['llm_pred_tones']
    .str.strip()
    .str.lower()
)

test_nfl_df_true['Media Tone'] = (
    test_nfl_df_true['Media Tone']
    .str.strip()
    .str.lower()
)


In [74]:
# Drop missing predictions
df_eval = test_nfl_df_true.dropna(subset=['llm_pred_tones'])

accuracy = accuracy_score(df_eval['Media Tone'], df_eval['llm_pred_tones'])
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(df_eval['Media Tone'], df_eval['llm_pred_tones']))

print("\nConfusion Matrix:")
print(confusion_matrix(df_eval['Media Tone'], df_eval['llm_pred_tones']))


Accuracy: 0.6545454545454545

Classification Report:
              precision    recall  f1-score   support

       mixed       1.00      0.08      0.15        12
    negative       0.66      0.83      0.73        23
    positive       0.64      0.80      0.71        20

    accuracy                           0.65        55
   macro avg       0.77      0.57      0.53        55
weighted avg       0.72      0.65      0.60        55


Confusion Matrix:
[[ 1  6  5]
 [ 0 19  4]
 [ 0  4 16]]


In [75]:
errors = df_eval[df_eval['Media Tone'] != df_eval['llm_pred_tones']]

errors[['Media Tone', 'llm_pred_tones', 'URL']].head(10)

,Media Tone,llm_pred_tones,URL
4,negative,positive,https://sports.yahoo.com/article/deion-sanders...
10,positive,negative,https://sports.yahoo.com/article/ben-dinucci-n...
19,negative,positive,https://www.usmagazine.com/entertainment/news/...
21,mixed,negative,https://sports.yahoo.com/articles/ex-nfl-playe...
27,mixed,negative,https://www.nfl.com/news/a-j-brown-let-frustra...
30,positive,negative,https://www.nbcsports.com/nfl/profootballtalk/...
31,mixed,negative,https://www.usmagazine.com/celebrity-news/news...
40,mixed,negative,https://sports.yahoo.com/twitter-justin-jeffer...
42,mixed,negative,https://sports.yahoo.com/nfl-cam-newton-trash-...
46,negative,positive,https://www.foxnews.com/sports/antonio-brown-s...


In [76]:
df_eval['match'] = df_eval['Media Tone'] == df_eval['llm_pred_tones']
agreement_rate = df_eval['match'].mean()

print("Agreement rate:", agreement_rate)


Agreement rate: 0.6545454545454545


In [77]:
errors['Media Tone'].value_counts()

Media Tone
mixed       11
negative     4
positive     4
Name: count, dtype: int64

Media Coverage

In [80]:
media_df_coverage = test_nfl_df_true.copy()

In [81]:
all_media_coverages = media_df_coverage['Media Coverage'].unique().tolist()

In [95]:
test_nfl_df_true.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL', 'website_text',
       'llm_pred', 'llm_signal_pred', 'llm_signal_rationale', 'llm_pred_tones',
       'llm_pred_coverages'],
      dtype='str')

In [103]:
import re

def extract_sources(row):
    raw = str(row.get("Sources", ""))

    # split by comma, semicolon, or newline
    parts = re.split(r",|;|\n", raw)

    # clean + normalize
    cleaned = [p.strip().lower() for p in parts if p.strip()]

    # remove duplicates
    return list(set(cleaned))

In [107]:
test_nfl_df_true['sources_list'] = test_nfl_df_true.apply(extract_sources, axis=1)

In [108]:
sources_exploded = test_nfl_df_true.explode("sources_list")

In [116]:
sources_exploded['sources_list'].value_counts().head()

sources_list
yahoo sports    24
us weekly        5
people           5
msn              4
fox news         4
Name: count, dtype: int64

In [114]:
# Based on journalistic authority and views (may not be very accurate classification)
HIGH_TIER = {
    "espn", "nfl.com", "cbssports", "cbs sports",
    "fox sports", "nbcsports", "sports illustrated",
    "the athletic", "yahoo sports",
    
    # general news
    "fox news", "cnn", "msnbc", "reuters", "associated press",
    "ap news", "usa today", "washington post", "nytimes",
    "wall street journal",

    # entertainment (important for NFL media ecosystem)
    "people", "entertainment weekly"
}

MEDIUM_TIER = {
    "bleacher report",
    "sporting news",
    "msn",
    "yahoo news",
    "us weekly",
    "the athetic"
}

In [123]:
def compute_source_strength(source_list):
    score = 0

    for s in source_list:
        if any(h in s for h in HIGH_TIER):
            score += 3
        elif any(m in s for m in MEDIUM_TIER):
            score += 1
        else:
            score -= 1  # unknown / low credibility sources

    return score

In [132]:
def classify_media_coverage_llm(row):

    event_type = row.get("llm_pred_event", "Unknown")
    sources_list = extract_sources(row)
    source_strength = compute_source_strength(sources_list)

    if not sources_list:
        return "Low"  # no sources = low coverage
    
    if event_type is None or event_type == "":
        event_type = "Unknown"

    prompt = f"""
You are classifying MEDIA COVERAGE of an NFL event.

You are estimating how widely the event was reported based on the event type and the strength of the sources. 

Inputs:

Event type: {event_type}
Source strength: {source_strength}

Source strength meaning:
- 1-3 -> Low coverage (few / weak sources)
- 4-6 -> Medium coverage 
- 7+ -> (many strong outlets)

Event types meaning:
- Controversy = more likely widely reported (increases coverage)
- Apology = moderate coverage
- Support = moderate coverage
- Reflection = lower coverage
- Humor = usually low unless strong sources

Task:
Use the inputs to decide coverage level.

Return ONLY one word:
High, Medium, or Low
No Explanation Included, just High, Medium, or Low
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"].strip()

In [133]:
# Run LLM classification
test_nfl_df_true['llm_pred_coverages'] = test_nfl_df_true.apply(classify_media_coverage_llm, axis=1)

In [134]:
y_true = test_nfl_df_true['Media Coverage']
y_pred = test_nfl_df_true['llm_pred_coverages']

In [135]:
test_nfl_df_true[['Media Coverage', 'llm_pred_coverages']].head(10)

,Media Coverage,llm_pred_coverages
1,medium,Medium
2,high,Medium
4,medium,Medium
5,medium,Medium
9,low,Low
10,medium,Low
11,low,Low
12,medium,Low
17,medium,Low
18,medium,Low


In [136]:
test_nfl_df_true['llm_pred_coverages'] = (
    test_nfl_df_true['llm_pred_coverages']
    .str.strip()
    .str.lower()
)

test_nfl_df_true['Media Coverage'] = (
    test_nfl_df_true['Media Coverage']
    .str.strip()
    .str.lower()
)

In [137]:
# Drop missing predictions
df_eval = test_nfl_df_true.dropna(subset=['llm_pred_coverages'])

accuracy = accuracy_score(df_eval['Media Coverage'], df_eval['llm_pred_coverages'])
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(df_eval['Media Coverage'], df_eval['llm_pred_coverages']))

print("\nConfusion Matrix:")
print(confusion_matrix(df_eval['Media Coverage'], df_eval['llm_pred_coverages']))

Accuracy: 0.6491228070175439

Classification Report:
              precision    recall  f1-score   support

        high       0.00      0.00      0.00         7
         low       0.47      0.53      0.50        15
      medium       0.72      0.83      0.77        35

    accuracy                           0.65        57
   macro avg       0.40      0.45      0.42        57
weighted avg       0.57      0.65      0.61        57


Confusion Matrix:
[[ 0  3  4]
 [ 0  8  7]
 [ 0  6 29]]


c:\Users\mayab\.virtualenvs\is310-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mayab\.virtualenvs\is310-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mayab\.virtualenvs\is310-env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

In [138]:
errors = df_eval[df_eval['Media Coverage'] != df_eval['llm_pred_coverages']]

errors[['Media Coverage', 'llm_pred_coverages', 'URL']].head(10)

,Media Coverage,llm_pred_coverages,URL
2,high,medium,https://www.cnn.com/2025/12/18/sport/football-...
10,medium,low,https://sports.yahoo.com/article/ben-dinucci-n...
12,medium,low,https://www.i24news.tv/en/news/international/s...
17,medium,low,https://sports.yahoo.com/articles/kyle-pitts-c...
18,medium,low,https://sports.yahoo.com/articles/shipped-off-...
21,medium,low,https://sports.yahoo.com/articles/ex-nfl-playe...
24,high,medium,https://people.com/vernon-davis-breaks-silence...
28,low,medium,https://broncoswire.usatoday.com/story/sports/...
29,low,medium,https://sports.yahoo.com/article/amik-robertso...
30,low,medium,https://www.nbcsports.com/nfl/profootballtalk/...


In [139]:
df_eval['match'] = df_eval['Media Coverage'] == df_eval['llm_pred_coverages']
agreement_rate = df_eval['match'].mean()

print("Agreement rate:", agreement_rate)

Agreement rate: 0.6491228070175439


In [140]:
errors['Media Coverage'].value_counts()

Media Coverage
high      7
low       7
medium    6
Name: count, dtype: int64

Goal: Use LLM to classify blind based on the url, website_text (the non-interpretable variables) - see if there are any converges or diverges from my classification as I would be the ground truth. 
- Use this source as a resource: https://cultureasdata-uiuc.github.io/is310-spring-2026/materials/interpreting-communicating-humanities-data/04-advanced-computational-methods.html#llm-classification-with-ollama

(Goal is to get this part done by Tuesday to see if this is feasible to scale up)

-------------------

In [6]:
nfl_df = nfl_df.drop(columns=['Unnamed: 13', 'Unnamed: 14'])

In [7]:
nfl_df.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL'],
      dtype='str')

In [8]:
nfl_df.shape

(263, 13)

In [9]:
nfl_cleaned = nfl_df.dropna()

In [10]:
nfl_cleaned.shape

(75, 13)

In [11]:
nfl_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 75 non-null     float64
 1   Player Name        75 non-null     str    
 2   Player Status      75 non-null     str    
 3   Year of Event      75 non-null     float64
 4   Statement Made     75 non-null     str    
 5   Event Description  75 non-null     str    
 6   Platform           75 non-null     str    
 7   Event Type         75 non-null     str    
 8   Media Tone         75 non-null     str    
 9   Post Status        75 non-null     str    
 10  Media Coverage     75 non-null     str    
 11  Sources            75 non-null     str    
 12  URL                75 non-null     str    
dtypes: float64(2), str(11)
memory usage: 34.7 KB


In [12]:
nfl_cleaned.dtypes

ID                   float64
Player Name              str
Player Status            str
Year of Event        float64
Statement Made           str
Event Description        str
Platform                 str
Event Type               str
Media Tone               str
Post Status              str
Media Coverage           str
Sources                  str
URL                      str
dtype: object

In [13]:
nfl_df['Event Type'].value_counts()

Event Type
Controversy    39
Support        16
Humor          10
Reflection      7
Apology         3
Name: count, dtype: int64

In [14]:
nfl_df['Media Tone'].value_counts()

Media Tone
Negative     33
Positive     25
Mixed        16
Positive      1
Name: count, dtype: int64

In [15]:
nfl_df['Platform'].value_counts().head()

Platform
Twitter/X                44
Instagram                10
St. Brown Podcast         1
"New Heights" podcast     1
"It's Giving" podcast     1
Name: count, dtype: int64

In [16]:
nfl_cleaned['Sources'].value_counts().head()

Sources
Yahoo Sports    11
ESPN             5
US Weekly        4
BroncosWire      2
MSN              2
Name: count, dtype: int64

In [17]:
nfl_cleaned.columns

Index(['ID', 'Player Name', 'Player Status', 'Year of Event', 'Statement Made',
       'Event Description', 'Platform', 'Event Type', 'Media Tone',
       'Post Status', 'Media Coverage', 'Sources', 'URL'],
      dtype='str')

In [18]:
nfl_cleaned.rename(columns={'ID': 'id', 'Player Name': 'player_name', 'Player Status': 'player_status', 'Year of Event': 'year_of_event', 
                                          'Statement Made': 'statement_made', 'Event Description': 'event_description', 'Platform':'platform',
                                          'Event Type': 'event_type', 'Media Tone': 'media_tone', 'Post Status': 'post_status', 
                                          'Media Coverage': 'media_coverage', 'Platform': 'platform', 'Sources': 'sources', 'URL': 'url'}, inplace=True)

In [19]:
{
  "ID": int,
  "Player Name": str,
  "Player Status": ["Playing", "Retired", "Free Agent"],
  "Year of Event": int,
  "Statement Made": str,
  "Event Description": str,
  "Platform": ["Twitter/X", "Instagram", "Podcast", "Press Conference", "Public Appearance"],
  "Event Type": ["Controversy", "Apology", "Support", "Reflection", "Humor"],
  "Media Tone": ["Positive", "Negative", "Mixed"],
  "Post Status": ["Active", "Deleted", "Available", "Unavailable"],
  "Media Coverage": ["Low", "Medium", "High"],
  "Sources": list[str]
}

{'ID': int,
 'Player Name': str,
 'Player Status': ['Playing', 'Retired', 'Free Agent'],
 'Year of Event': int,
 'Statement Made': str,
 'Event Description': str,
 'Platform': ['Twitter/X',
  'Instagram',
  'Podcast',
  'Press Conference',
  'Public Appearance'],
 'Event Type': ['Controversy', 'Apology', 'Support', 'Reflection', 'Humor'],
 'Media Tone': ['Positive', 'Negative', 'Mixed'],
 'Post Status': ['Active', 'Deleted', 'Available', 'Unavailable'],
 'Media Coverage': ['Low', 'Medium', 'High'],
 'Sources': list[str]}

In [20]:
nfl_cleaned.columns

Index(['id', 'player_name', 'player_status', 'year_of_event', 'statement_made',
       'event_description', 'platform', 'event_type', 'media_tone',
       'post_status', 'media_coverage', 'sources', 'url'],
      dtype='str')

In [21]:
nfl_cleaned["event_type"] = nfl_cleaned["event_type"].str.strip()
nfl_cleaned["media_tone"] = nfl_cleaned["media_tone"].str.strip()
nfl_cleaned["platform"] = nfl_cleaned["platform"].str.strip()
nfl_cleaned["player_status"] = nfl_cleaned["player_status"].str.strip()

In [22]:
event_types = ["Controversy", "Apology", "Support", "Reflection", "Humor"]
media_tones = ["Positive", "Negative", "Mixed"]
platform = ["Twitter/X", "Instagram", "Podcast", "Press Conference", "Public Appearance"]
player_status = ["Playing", "Retired", "Free Agent"]
media_coverage = ["Low", "Medium", "High"]
post_status = ["Active", "Deleted", "Available", "Unavailable"]

Creating dictionaries to generate patterns to follow in interpretative variables

In [8]:
event_type_rules = {
    "Controversy": ["criticized", "backlash", "accused", "under fire"],
    "Apology": ["apologized", "regret", "sorry", "mistake"],
    "Support": ["supported", "defended", "donated", "stood by"],
    "Reflection": ["career", "retirement", "looking back", "legacy"],
    "Humor": ["joked", "sarcastic", "funny", "playful"]
}

In [9]:
media_tone_rules = {
    "Positive": ["praised", "celebrated", "highlighted"],
    "Negative": ["criticized", "controversial", "under fire"],
    "Mixed": ["debated", "mixed reactions", "split opinions"]
}

In [10]:
media_coverage_rules = {
    "High": [
        "widely reported", "major outlets", "headline", "viral",
        "extensive coverage", "national attention", "breaking news"
    ],
    "Medium": [
        "reported by several outlets", "moderate attention",
        "covered by sports media", "notable coverage"
    ],
    "Low": [
        "briefly mentioned", "limited coverage",
        "few outlets", "minor attention", "local report"
    ]
}

In [11]:
high_tier_outlets = [
    "ESPN", "NFL Network", "Yahoo Sports", "Fox Sports", "CBS Sports"
]

mid_tier_outlets = [
    "Bleacher Report", "SB Nation", "Sports Illustrated"
]

low_tier_outlets = [
    "local news", "team website", "small blog"
]

In [23]:
def classify_coverage(row):
    text = (str(row["event_description"]) + " " + str(row["sources"])).lower()
    
    score = 0
    
    # keyword signals
    for word in media_coverage_rules["High"]:
        if word in text:
            score += 2
            
    for word in media_coverage_rules["Medium"]:
        if word in text:
            score += 1
            
    for word in media_coverage_rules["Low"]:
        if word in text:
            score -= 1
    
    # outlet signals
    for outlet in high_tier_outlets:
        if outlet.lower() in text:
            score += 2
            
    for outlet in mid_tier_outlets:
        if outlet.lower() in text:
            score += 1
            
    # event-type boost (optional but realistic)
    if row["event_type"] == "Controversy":
        score += 1
    
    # final classification
    if score >= 3:
        return "High"
    elif score >= 1:
        return "Medium"
    else:
        return "Low"

In [28]:
platform_dict = {
    "Twitter/X": ["tweet", "X post", "retweeted"],
    "Instagram": ["Instagram post", "story", "caption"],
    "Podcast": ["podcast", "interview episode"],
    "Press Conference": ["press conference", "media availability"],
    "Public Appearance": ["event", "appearance", "ceremony"]
}

Get seed examples

In [30]:
seed_examples = nfl_cleaned.sample(8).to_dict(orient='records')
print(seed_examples[0])

{'id': 66.0, 'player_name': 'Josh Allen', 'player_status': 'Playing', 'year_of_event': 2023.0, 'statement_made': "I saw some stuff on Twitter and people should not be attacking him whatsoever. I'm glad that Damar's family came out and said that. ", 'event_description': 'Comes to the defense of Tee Higgins, who received hate from tackle leading to Hamlin to being hospitalized', 'platform': 'Post-practice news conference', 'event_type': 'Support', 'media_tone': 'Positive', 'post_status': 'Available', 'media_coverage': 'Low', 'sources': 'Cincinnati Enquirer', 'url': 'https://www.cincinnati.com/story/sports/nfl/bengals/2023/01/05/josh-allen-people-should-not-be-attacking-tee-higgins-whatsoever-as-damar-hamlin-recovers-cincinnati/69782959007/?gnt-cfr=1&gca-cat=p&gca-uir=true&gca-epti=z115240e1162xxv115240d--58--b--58--&gca-ft=200&gca-ds=sophi'}


-----

In [31]:
def build_prompt(seed_examples_batch):
    return f"""
You are generating structured NFL media dataset records.

TASK:
Generate 20 NEW rows of a dataset called nfl_media.

You MUST follow this schema exactly:
- ID (integer)
- Player Name (NFL player)
- Player Status (Playing, Retired, Free Agent)
- Year of Event (2020–2026)
- Statement Made (realistic quote or paraphrased statement)
- Event Description (context of event)
- Platform (Twitter/X, Instagram, Podcast, Press Conference, Public Appearance)
- Event Type (Controversy, Apology, Support, Reflection, Humor)
- Media Tone (Positive, Negative, Mixed)
- Post Status (Active, Deleted, Available, Unavailable)
- Media Coverage (Low, Medium, High)
- Sources (sports/media outlets)
- URL (realistic placeholder)

RULES:
- Be realistic (NFL-related only)
- Do NOT repeat identical events
- Maintain diversity in players and platforms
- Event Type must match Statement tone
- Media Tone must match framing of event
- Avoid exaggerated or fictional drama
- Keep consistent sports journalism style

EVENT TYPE RULES:
Controversy → criticism, backlash, accusations
Apology → remorse, apology, regret
Support → defending others, solidarity
Reflection → career thoughts, retirement, past review
Humor → joking, sarcasm, playful remarks

MEDIA TONE RULES:
Positive → praise, celebration
Negative → criticism, backlash framing
Mixed → balanced reporting

HERE ARE REAL EXAMPLES FROM MY DATASET:
{seed_examples_batch}

OUTPUT FORMAT:
Return ONLY valid JSON list of 20 objects.

Do NOT include explanations.
"""

In [32]:
prompt = build_prompt(seed_examples)
print(prompt)


You are generating structured NFL media dataset records.

TASK:
Generate 20 NEW rows of a dataset called nfl_media.

You MUST follow this schema exactly:
- ID (integer)
- Player Name (NFL player)
- Player Status (Playing, Retired, Free Agent)
- Year of Event (2020–2026)
- Statement Made (realistic quote or paraphrased statement)
- Event Description (context of event)
- Platform (Twitter/X, Instagram, Podcast, Press Conference, Public Appearance)
- Event Type (Controversy, Apology, Support, Reflection, Humor)
- Media Tone (Positive, Negative, Mixed)
- Post Status (Active, Deleted, Available, Unavailable)
- Media Coverage (Low, Medium, High)
- Sources (sports/media outlets)
- URL (realistic placeholder)

RULES:
- Be realistic (NFL-related only)
- Do NOT repeat identical events
- Maintain diversity in players and platforms
- Event Type must match Statement tone
- Media Tone must match framing of event
- Avoid exaggerated or fictional drama
- Keep consistent sports journalism style

EVENT

In [33]:
with open("batch1.json") as f:
    batch1 = json.load(f)

batch_df = pd.DataFrame(batch1)

batch_df.head()

,id,player_name,player_status,year_of_event,statement_made,event_description,platform,event_type,media_tone,post_status,media_coverage,sources,url
0,101,Patrick Mahomes,Playing,2024,"We didn't execute the way we expect to, and th...",Postgame comments after a regular season loss ...,Press Conference,Reflection,Mixed,Available,High,"ESPN, NFL Network",https://www.espn.com/nfl/story/_/id/39186641/c...
1,102,Jalen Hurts,Playing,2023,"It's about how we respond, not what happens.",Comments following a tough loss emphasizing te...,Press Conference,Reflection,Positive,Available,High,"NBC Sports, ESPN",https://www.nbcsports.com/nfl/profootballtalk/...
2,103,Odell Beckham Jr.,Playing,2021,I just want to be somewhere I'm appreciated an...,Social media post during tensions with the Cle...,Instagram,Controversy,Negative,Deleted,High,"Bleacher Report, ESPN",https://www.espn.com/nfl/story/_/id/32543974/c...
3,104,Russell Wilson,Playing,2022,I take full responsibility for how I played to...,Addressing criticism after a poor performance ...,Press Conference,Apology,Mixed,Available,High,"Fox Sports, ESPN",https://www.foxsports.com/stories/nfl/russell-...
4,105,Travis Kelce,Playing,2024,"Man, I gotta stop celebrating like I'm 21 ðŸ˜‚",Joking about his touchdown celebrations after ...,Podcast,Humor,Positive,Available,Medium,"New Heights Podcast, Yahoo Sports",https://www.youtube.com/@newheightshow


In [34]:
def validate_row(row):
    return (
        row["event_type"] in ["Controversy","Apology","Support","Reflection","Humor"]
        and row["media_tone"] in ["Positive","Negative","Mixed"]
        and row["media_coverage"] in ["Low","Medium","High"]
    )

batch_df["valid"] = batch_df.apply(validate_row, axis=1)

print(batch_df["valid"].value_counts())

valid
True    20
Name: count, dtype: int64


In [35]:
nfl_combined = pd.concat([nfl_cleaned, batch_df], ignore_index=True)

print(nfl_combined.shape)

(95, 14)


In [ ]:
all_batches = [nfl_combined]

for i in tqdm(range(1, 30)):  # ~30 batches
    
    input(f"\nGenerate batch {i+1} in ChatGPT and save as batch{i+1}.json, then press Enter...")
    
    with open(f"batch{i+1}.json", encoding="utf-8") as f:
        batch = json.load(f)
    
    batch_df = pd.DataFrame(batch)
    
    # validate
    batch_df = batch_df[
        batch_df["event_type"].isin(["Controversy","Apology","Support","Reflection","Humor"])
        & batch_df["media_tone"].isin(["Positive","Negative","Mixed"])
        & batch_df["media_coverage"].isin(["Low","Medium","High"])
    ]
    
    all_batches.append(batch_df)

final_df = pd.concat(all_batches, ignore_index=True)

  0%|          | 0/29 [00:00<?, ?it/s]

In [ ]:
final_df = final_df.drop_duplicates(subset=["statement_made"])

final_df = final_df.reset_index(drop=True)
final_df['id'] = range(1, len(final_df) + 1)

final_df.to_csv("nfl_media_expanded.csv", index=False)

print(final_df.shape)

(665, 14)


In [3]:
final_df = pd.read_csv("nfl_media_expanded.csv")

In [4]:
final_df.head()

,id,player_name,player_status,year_of_event,statement_made,event_description,platform,event_type,media_tone,post_status,media_coverage,sources,url,valid
0,1,Jason Kelce,Retired,2025.0,Man I love the 4th...we all share in common th...,Instagram post celebrating July 4th sparked ba...,Instagram,Controversy,Negative,Active,High,"Fox News, US Weekly, Yahoo Sports",https://www.yahoo.com/entertainment/articles/j...,NaN
1,2,Warren Sapp,Retired,2025.0,Texas is Fake Football,Response to a tweet that praises Texas running...,Twitter/X,Controversy,Negative,Deleted,Medium,"MSN, Yahoo Sports",https://sports.yahoo.com/articles/nfl-legend-w...,NaN
2,3,Puka Nacua,Playing,2025.0,I deeply apologize...I do not stand for any fo...,Apologizes after making antisemitic gesture du...,Instagram,Apology,Negative,Deleted,High,"CNN, Yahoo Sports, BBC",https://www.cnn.com/2025/12/18/sport/football-...,NaN
3,4,Travis Kelce,Playing,2024.0,happy easter...#shoutout to Jesus for takin on...,Wishing everyone a Happy Easter while making j...,Twitter/X,Controversy,Mixed,Active,High,"Yahoo Sports, E! News, People",https://www.yahoo.com/entertainment/articles/t...,NaN
4,5,Deion Sanders,Playing,2024.0,He will be a top 5 pick. Where yo son going?,Response to a fan about the draft pick ranking...,Twitter/X,Controversy,Negative,Deleted,Medium,"Yahoo Sports, ESPN, Fox News",https://sports.yahoo.com/article/deion-sanders...,NaN


In [5]:
final_df['id'].duplicated().sum()

np.int64(0)

Interpretative portion - will use the dictionaries I created to label the interpretative variables and compare to AI's inputs

In [6]:
def classify_event_type(text):
    text = str(text).lower()
    
    for label, keywords in event_type_rules.items():
        if any(k in text for k in keywords):
            return label
    
    return "Reflection"

In [13]:
final_df["event_type_rule"] = final_df["statement_made"].apply(classify_event_type)

In [15]:
final_df["event_type"].value_counts()

event_type
Support        319
Reflection     154
Controversy    110
Humor           54
Apology         28
Name: count, dtype: int64

In [14]:
final_df["event_type_rule"].value_counts()

event_type_rule
Reflection    657
Apology         8
Name: count, dtype: int64

In [16]:
def classify_media_tone(text):
    text = str(text).lower()
    
    for label, keywords in media_tone_rules.items():
        if any(k in text for k in keywords):
            return label
    
    return "Mixed"

In [19]:
final_df["media_tone_rule"] = final_df["statement_made"].apply(classify_media_tone)

In [21]:
final_df['media_tone'].value_counts()

media_tone
Positive    473
Mixed       107
Negative     85
Name: count, dtype: int64

In [20]:
final_df['media_tone_rule'].value_counts()

media_tone_rule
Mixed       664
Negative      1
Name: count, dtype: int64

Have to refine the event type rules and media tone rules - maybe look at the common words used in the 'text' variable

In [29]:
final_df.columns

Index(['id', 'player_name', 'player_status', 'year_of_event', 'statement_made',
       'event_description', 'platform', 'event_type', 'media_tone',
       'post_status', 'media_coverage', 'sources', 'url', 'valid',
       'event_type_rule', 'media_type_rule', 'media_tone_rule',
       'media_coverage_rule'],
      dtype='str')

In [33]:
final_df['player_name'].value_counts().head()

player_name
Travis Kelce       7
Patrick Mahomes    7
Tyreek Hill        7
Caleb Williams     6
Saquon Barkley     6
Name: count, dtype: int64

In [30]:
final_df['statement_made']

0      Man I love the 4th...we all share in common th...
1                                 Texas is Fake Football
2      I deeply apologize...I do not stand for any fo...
3      happy easter...#shoutout to Jesus for takin on...
4           He will be a top 5 pick. Where yo son going?
                             ...                        
660    I’m here to play physical football. The Bills ...
661    I’m looking forward to learning from the guys ...
662    When the work is put in, the decisions speak f...
663    I fit this defense. Chicago wants toughness an...
664    Grateful for the chance through the Internatio...
Name: statement_made, Length: 665, dtype: str

In [24]:
final_df["media_coverage_rule"] = final_df.apply(classify_coverage, axis=1)

In [26]:
final_df['media_coverage'].value_counts()

media_coverage
Medium    347
High      194
Low       124
Name: count, dtype: int64

In [25]:
final_df["media_coverage_rule"].value_counts()

media_coverage_rule
Low       349
Medium    184
High      132
Name: count, dtype: int64